# Capstone — mirrors your deployed research paper

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Question

*The research question and the decision it supports.*

In [138]:
# ## 1. Question

# ### Research Question

# Which content pages should be prioritized for refresh based on their observed search visibility, engagement, performance trends, and content freshness signals?

# ### Decision Supported

# This analysis supports content teams in deciding which pages should be reviewed and potentially refreshed first.

# ### Why This Matters

# Content teams have limited time and resources, so a ranked list of refresh opportunities can help prioritize pages for review.

# ### Decision Risk

# A poor prioritization decision may cause the team to spend resources refreshing pages with limited opportunity while overlooking pages that show stronger signals for review.

# ### Expected Output

# The final output is a ranked list of content pages with a refresh opportunity score, reason codes, and recommended actions.



## 2. Data

*Which release, which tables, date windows, what you excluded and why. Public-safe.*

In [139]:
import pandas as pd
import numpy as np

url = "https://raw.githubusercontent.com/likithagarlapati7-oss/flyrank-ml-internship/main/data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(url)

print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

Shape: (30000, 44)

Columns:
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']


In [140]:
print("Dataset shape:", df.shape)
print("Unique content pages:", df["content_id"].nunique())
print("Unique clients:", df["client_id"].nunique())

Dataset shape: (30000, 44)
Unique content pages: 30000
Unique clients: 32


In [141]:
print("Number of unique content pages:", df["content_id"].nunique())
print("Number of unique clients:", df["client_id"].nunique())

print("\nMissing values:")
print(df.isnull().sum().sort_values(ascending=False).head(20))

Number of unique content pages: 30000
Number of unique clients: 32

Missing values:
provider_used        21438
word_count            7699
char_count            7699
word_count_tier       7699
char_count_tier       7699
model_used            5733
trend_pct             3388
competition_level     2610
search_volume         2468
cpc                   2468
competition           2468
main_intent           2374
scroll_rate            125
content_type             0
client_id                0
content_id               0
impressions_90d          0
clicks_90d               0
pageviews_90d            0
sessions_90d             0
dtype: int64


In [142]:
missing = df.isnull().sum()

missing = missing[missing > 0].sort_values(ascending=False)

print("Columns with missing values:")
print(missing)

Columns with missing values:
provider_used        21438
word_count_tier       7699
char_count            7699
word_count            7699
char_count_tier       7699
model_used            5733
trend_pct             3388
competition_level     2610
search_volume         2468
competition           2468
cpc                   2468
main_intent           2374
scroll_rate            125
dtype: int64


In [143]:
print("Duplicate rows:", df.duplicated().sum())
print("Duplicate content IDs:", df["content_id"].duplicated().sum())

Duplicate rows: 0
Duplicate content IDs: 0


In [144]:
df.describe().T

,count,mean,std,min,25%,50%,75%,max
search_volume,27532.0,158.882391,1518.270825,0.0,0.0,10.00,20.00,74000.00
competition,27532.0,0.146954,0.285241,0.0,0.0,0.00,0.13,1.00
cpc,27532.0,0.485342,2.101560,0.0,0.0,0.00,0.00,100.36
word_count,22301.0,3107.760325,1452.382598,8.0,2413.0,2877.00,3666.00,9546.00
char_count,22301.0,20665.277835,10115.344042,40.0,15644.0,19116.00,24011.00,111158.00
impressions_90d,30000.0,5200.366300,16838.019547,1.0,81.0,731.00,3615.25,517715.00
clicks_90d,30000.0,16.097333,75.076958,0.0,0.0,1.00,7.00,4178.00
pageviews_90d,30000.0,49.942467,152.101430,0.0,2.0,8.00,33.00,5998.00
sessions_90d,30000.0,37.066633,107.069131,1.0,2.0,7.00,27.00,4345.00
users_90d,30000.0,35.937700,103.748185,1.0,2.0,7.00,27.00,4913.00


In [145]:
## 2. Data

# The analysis uses the anonymized FlyRank ML Internship dataset available through the project repository.

# The original working CSV contains 30,000 content-page records and 44 columns. Each row represents one anonymized content page.

# The dataset contains search-performance, engagement, content freshness, content characteristics, and recent performance variables.

# ### Main Signal Groups

# - Search visibility: impressions, search volume, and average position
# - Engagement: sessions, engaged sessions, engagement rate, and scroll rate
# - Click performance: clicks and CTR
# - Freshness: content age and days since last update
# - Recent performance: impressions, clicks, and sessions for the latest 30-day period compared with the previous 30-day period

# ### Derived Fields

# For the analysis, percentage-change features are calculated from the recent and previous 30-day performance fields. A proxy `refresh_opportunity` label is then created for prioritization analysis.

# ### Exclusions

# Client-identifying information, private queries, domains, credentials, and raw private exports are not used in the analysis.

# The anonymized content and client identifiers are retained only where necessary for identifying rows in the ranked output.

# ### Data Quality

# The dataset contains 30,000 unique content pages and 32 anonymized clients. No duplicate rows or duplicate content IDs were found.

# Several fields contain missing values, including provider information, content-length fields, search-volume fields, and trend metrics. Missing rows are handled during model preparation rather than being silently filled.

# ### Data Limitations

# The working CSV is an aggregated dataset rather than a complete page-level time series. It contains 90-day aggregates and comparisons between the latest and previous 30-day periods, but it does not provide an explicit observation date.

# Therefore, this analysis does not claim to establish causal effects of refreshing content.

## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*

In [146]:
df["impression_change_pct"] = (
    (df["impressions_last_30d"] - df["impressions_prev_30d"])
    / (df["impressions_prev_30d"] + 1)
) * 100

df["click_change_pct"] = (
    (df["clicks_last_30d"] - df["clicks_prev_30d"])
    / (df["clicks_prev_30d"] + 1)
) * 100

df["session_change_pct"] = (
    (df["sessions_last_30d"] - df["sessions_prev_30d"])
    / (df["sessions_prev_30d"] + 1)
) * 100

In [147]:
df["refresh_opportunity"] = (
    (
        (df["impression_change_pct"] < df["impression_change_pct"].median()) &
        (df["click_change_pct"] < df["click_change_pct"].median())
    )
    |
    (
        (df["content_age_days"] > df["content_age_days"].median()) &
        (df["engagement_rate"] < df["engagement_rate"].median())
    )
).astype(int)

print(df["refresh_opportunity"].value_counts())

refresh_opportunity
0    25849
1     4151
Name: count, dtype: int64


In [148]:
### Label Definition

# The dataset does not contain a ground-truth field indicating whether a page was actually refreshed.

# Therefore, this study uses a proxy refresh-opportunity label. A page is considered a higher-opportunity candidate when it shows combinations of weaker recent performance and/or older content with weaker engagement.

# This label represents a prioritization heuristic rather than an observed business outcome.

In [149]:
features = [
    "impressions_90d",
    "clicks_90d",
    "pageviews_90d",
    "sessions_90d",
    "engaged_sessions_90d",
    "search_volume",
    "avg_position",
    "days_since_last_update",
    "word_count",
    "char_count",
    "scroll_rate",
    "ai_traffic_pct"
]

In [150]:
model_df = df[features + ["refresh_opportunity"]].copy()

model_df = model_df.replace(
    [np.inf, -np.inf],
    np.nan
)

model_df = model_df.dropna()

X = model_df[features]
y = model_df["refresh_opportunity"]

print("X:", X.shape)
print("y:", y.shape)

X: (19897, 12)
y: (19897,)


In [151]:
baseline_score = df.loc[X.index, "days_since_last_update"].values

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=10,
    min_samples_leaf=5,
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train, y_train)

In [ ]:
def precision_at_k(y_true, scores, k):
    order = np.argsort(scores)[::-1][:k]
    return np.mean(np.array(y_true)[order])

model_scores = rf.predict_proba(X_test)[:, 1]

for k in [10, 20, 50, 100]:
    score = precision_at_k(
        y_test.reset_index(drop=True),
        model_scores,
        k
    )

    print(f"Precision@{k}: {score:.3f}")

In [ ]:
### Leakage Check

# The original refresh-opportunity target was constructed from recent performance changes, content age, and engagement signals. Including those exact variables in the model would allow the model to reproduce the proxy rule rather than provide an independent ranking signal.

# For the revised analysis, the direct target-construction variables `impression_change_pct`, `click_change_pct`, `session_change_pct`, `engagement_rate`, and `content_age_days` are excluded from the model feature set.

# `days_since_last_update` is retained as an observed freshness signal and is also used as the simple baseline.

# The evaluation therefore tests whether the remaining observed page characteristics can identify pages matching the proxy opportunity definition.

## 4. Results (vs baseline)

*Model vs baseline on the same split. The honest table.*

In [ ]:
# ### 4. Results

# The Random Forest model and the baseline are evaluated on the same held-out test set.

# Precision@K is used as the primary evaluation metric because the intended use is to prioritize a limited number of pages for human review.

# The results measure how well the model identifies pages matching the proxy opportunity definition. They should not be interpreted as evidence that refreshing the recommended pages will necessarily improve traffic, clicks, rankings, or engagement.

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score

y_pred = rf.predict(X_test)

print("Random Forest")
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1:", f1_score(y_test, y_pred))

print("\nPrecision@K")

for k in [10, 20, 50, 100]:
    score = precision_at_k(
        y_test.reset_index(drop=True),
        model_scores,
        k
    )
    print(f"Precision@{k}: {score:.3f}")

In [ ]:
print("\nBaseline Precision@K")

baseline_test_scores = X_test["days_since_last_update"].values

for k in [10, 20, 50, 100]:
    score = precision_at_k(
        y_test.reset_index(drop=True),
        baseline_test_scores,
        k
    )
    print(f"Precision@{k}: {score:.3f}")

## 5. Limitations

*What this work cannot claim.*

In [ ]:
## 5. Limitations

# This analysis has several important limitations.

# 1. The target is a proxy for refresh opportunity rather than a record of actual content refresh outcomes.

# 2. The model identifies pages with characteristics associated with the defined opportunity signal. It does not establish that refreshing a page will cause higher traffic, clicks, rankings, or engagement.

# 3. The working dataset is aggregated and does not provide a complete experimental history of page refreshes and their outcomes.

# 4. Historical performance patterns may not represent future search behavior.

# 5. Missing or incomplete analytics can affect individual recommendations.

# 6. The model should therefore be used as decision support for content review rather than as an automatic refresh decision system.

# 7. The analysis does not claim to explain or predict Google's ranking algorithm.

## 6. Ranked recommendations

*The action playbook output — the paper's recommendations section.*

In [ ]:
scored_pages = model_df.copy()

scored_pages["opportunity_score"] = rf.predict_proba(
    scored_pages[features]
)[:, 1]

scored_pages = scored_pages.sort_values(
    "opportunity_score",
    ascending=False
)

scored_pages["rank"] = range(1, len(scored_pages) + 1)

recommendations = df.loc[scored_pages.index].copy()
recommendations["opportunity_score"] = scored_pages["opportunity_score"]
recommendations["rank"] = scored_pages["rank"]

In [ ]:
# Create reason codes

recommendations["reason"] = np.where(
    recommendations["trend_pct"] < 0,
    "DECLINING_TREND",
    np.where(
        recommendations["avg_position"] > df["avg_position"].median(),
        "WEAKER_POSITION",
        np.where(
            recommendations["ctr"] < df["ctr"].median(),
            "LOW_CTR",
            np.where(
                recommendations["days_since_last_update"] >
                df["days_since_last_update"].median(),
                "STALE_CONTENT",
                "GENERAL_REVIEW"
            )
        )
    )
)

# Create recommended actions

recommendations["recommended_action"] = recommendations["reason"].map({
    "DECLINING_TREND":
        "Review declining sections and update content relevance",
    "WEAKER_POSITION":
        "Review search intent coverage and content depth",
    "LOW_CTR":
        "Review title, metadata, and content messaging",
    "STALE_CONTENT":
        "Review and update outdated content",
    "GENERAL_REVIEW":
        "Perform general content quality review"
})

# Check that the columns now exist
print("Reason column:", "reason" in recommendations.columns)
print("Action column:", "recommended_action" in recommendations.columns)

print(
    recommendations[
        ["content_id", "opportunity_score", "reason", "recommended_action"]
    ].head()
)

In [ ]:
top20 = recommendations[
    [
        "rank",
        "content_id",
        "content_type",
        "opportunity_score",
        "reason",
        "recommended_action",
        "search_volume",
        "impressions_90d",
        "clicks_90d",
        "ctr",
        "avg_position",
        "content_age_days",
        "days_since_last_update",
        "trend_pct"
    ]
].head(20)

top20

## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*

In [ ]:
importance = pd.DataFrame({
    "feature": features,
    "importance": rf.feature_importances_
}).sort_values("importance", ascending=False)

importance

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))

plt.barh(
    importance["feature"],
    importance["importance"]
)

plt.xlabel("Importance")
plt.ylabel("Feature")
plt.title("Random Forest Feature Importance")
plt.gca().invert_yaxis()

plt.show()

In [ ]:
top20

In [ ]:
plt.figure(figsize=(8, 5))

plt.hist(model_scores, bins=20)

plt.xlabel("Refresh Opportunity Score")
plt.ylabel("Number of Pages")
plt.title("Distribution of Refresh Opportunity Scores")

plt.show()

In [ ]:
# Save final artifacts

top20.to_csv(
    "top_20_refresh_recommendations.csv",
    index=False
)

importance.to_csv(
    "feature_importance.csv",
    index=False
)

print("Saved:")
print("- top_20_refresh_recommendations.csv")
print("- feature_importance.csv")

In [ ]:
results = []

for k in [10, 20, 50, 100]:
    model_precision = precision_at_k(
        y_test.reset_index(drop=True),
        model_scores,
        k
    )

    baseline_precision = precision_at_k(
        y_test.reset_index(drop=True),
        baseline_test_scores,
        k
    )

    results.append({
        "k": k,
        "model_precision_at_k": model_precision,
        "baseline_precision_at_k": baseline_precision
    })

results_df = pd.DataFrame(results)

results_df

In [ ]:
results_df.to_csv(
    "model_vs_baseline_results.csv",
    index=False
)

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.